# Simplified hierarchical morphotope classification (HiMoC)

In [ ]:
import geopandas as gpd
import libpysal
import matplotlib.pyplot as plt
import momepy
import shapely
import neatnet

Pick a place, ideally a town with a good coverage in OpenStreetMap and its local CRS.

In [ ]:
place = "Norwich, UK"
epsg_uk = 27700

In [ ]:
# norwich_centre = gpd.tools.geocode(place)

norwich_centre = gpd.GeoSeries(
    [shapely.Point(1.29626, 52.62858)], name="geometry", crs="epsg:4326"
)

In [ ]:
norwich_radius = norwich_centre.to_crs(epsg=epsg_uk).buffer(2_000)

In [ ]:
from overturemaps import record_batch_reader

In [ ]:
reader = record_batch_reader(
    "building", bbox=tuple(norwich_radius.to_crs(epsg=4326).bounds.values[0])
)

In [ ]:
res = reader.read_all()

In [ ]:
import geopandas as gpd

buildings = gpd.GeoDataFrame.from_arrow(res)
buildings = buildings.set_crs(epsg=4326)

In [ ]:

unwanted = ['garage',
 'garages',
 'shed',
 'bungalow',
 'roof',
 'parking',
 'kiosk',
 'hut',
 'pavilion',
 'greenhouse',
 'cabin',
 'toilets',
 'farm_auxiliary',
 'guardhouse']

In [ ]:
buildings = buildings[~buildings['class'].isin(unwanted)]

### Buildings

In [ ]:
buildings = buildings[buildings.geom_type == "Polygon"]

And we should re-project the data from WGS84 to the local projection in meters (momepy default values assume meters not feet or degrees). We will also drop unnecessary columns.

In [ ]:
buildings = buildings[["geometry"]].to_crs(epsg=27700)


In [ ]:
buildings = buildings[buildings.area > 25].reset_index(drop=True)
buildings.head()

In [ ]:
from libpysal.graph import Graph

In [ ]:
bgraph = Graph.build_fuzzy_contiguity(buildings.geometry, buffer=.5)

In [ ]:
buildings

In [ ]:
cbuildings = buildings.groupby(bgraph.component_labels).apply(lambda x: x.geometry.union_all())
cbuildings = gpd.GeoSeries(cbuildings, name='geometry', crs=buildings.crs)

In [ ]:
overture_buildings = buildings.copy()

In [ ]:
# buildings = cbuildings.to_frame()

In [ ]:
buildings.shape

### Streets

Similar operations are done with streets.

In [ ]:
reader = record_batch_reader(
    "segment", bbox=tuple(norwich_radius.to_crs(epsg=4326).bounds.values[0])
)

In [ ]:
res = reader.read_all()

In [ ]:
streets = gpd.GeoDataFrame.from_arrow(res)
streets = streets.set_crs(epsg=4326)
streets

We can also do some preprocessing using momepy to ensure we have proper network topology.

In [ ]:
## service road removed
approved_roads = [
    "living_street",
    "motorway",
    "motorway_link",
    "pedestrian",
    "primary",
    "primary_link",
    "residential",
    "secondary",
    "secondary_link",
    "tertiary",
    "tertiary_link",
    "trunk",
    "trunk_link",
    "unclassified",
]
streets = streets[streets["class"].isin(approved_roads)]

In [ ]:
def to_drop_tunnel(row):
    """Find whether or not a road segment has a tunnel thats more than 50 metres."""
    tunnel_length = row.geometry.length
    flags = row.road_flags

    total_tunnel_proportion = -1
    for flag in flags:
        if "values" in flag and ("is_tunnel" in flag["values"]):
            # between could be missing to show the whole thing is a tunnel
            total_tunnel_proportion = (
                0.0 if total_tunnel_proportion < 0 else total_tunnel_proportion
            )
            # betweencould be None to indicate the whole thing is a tunnel
            if ("between" in flag) and (flag["between"] is not None):
                s, e = flag["between"][0], flag["between"][1]
                total_tunnel_proportion += e - s

    if (
        total_tunnel_proportion * tunnel_length
    ) > 50 or total_tunnel_proportion == 0.0:
        return True
    return False

In [ ]:
## drop tunnels
to_filter = streets.loc[~streets.road_flags.isna(),].to_crs(epsg=epsg_uk)
tunnels_to_drop = to_filter.apply(to_drop_tunnel, axis=1)
streets = streets.drop(to_filter[tunnels_to_drop].index)

In [ ]:
streets = streets.to_crs(epsg=epsg_uk)
streets = streets.sort_values("id")[["id", "geometry", "class"]].reset_index(
    drop=True
)

In [ ]:
## simplify
simplified = neatnet.neatify(
    streets,
    exclusion_mask=buildings.geometry,
    artifact_threshold_fallback=7,
)

In [ ]:
streets = simplified.copy()

In [ ]:
corine = gpd.read_file('../../../data/corine/data/clc2018_uk.shp')

In [ ]:
# blgs, land = corine.sindex.query(buildings.geometry, predicate='intersects')

In [ ]:
land, blgs = buildings.sindex.query(corine.geometry, predicate='intersects')

In [ ]:
buildings['landuse'] = None
buildings.iloc[blgs, -1] = corine.iloc[land, 4]

In [ ]:
buildings = buildings[buildings.landuse.str.startswith('11')].reset_index(drop=True)

## Generated data

### Tessellation

Given building footprints:

In [ ]:
import pandas as pd
import shapely

In [ ]:
idxs = (
    pd.DataFrame(shapely.get_coordinates(buildings.representative_point()))
    .drop_duplicates()
    .index
)
limit = momepy.buffered_limit(buildings.loc[idxs], "adaptive")

In [ ]:
tessellation = momepy.morphological_tessellation(buildings, clip=limit)

OpenStreetMap data are often problematic due to low quality of some polygons. If some collapse, we get a mismatch between the length of buildings and the length of polygons.

In [ ]:
collapsed, _ = momepy.verify_tessellation(tessellation, buildings)

In [ ]:
idxs = (
    pd.DataFrame(shapely.get_coordinates(buildings.representative_point()))
    .drop_duplicates()
    .index
)

Better to drop affected buildings and re-create tessellation.

In [ ]:
buildings = buildings.drop(collapsed).reset_index(drop=True)
idxs = (
    pd.DataFrame(shapely.get_coordinates(buildings.representative_point()))
    .drop_duplicates()
    .index
)
limit = momepy.buffered_limit(buildings.iloc[idxs], "adaptive")

tessellation = momepy.morphological_tessellation(buildings, clip=limit)

Check the result.

In [ ]:
tessellation.shape[0] == buildings.shape[0]

### Link streets

Link unique IDs of streets to buildings and tessellation cells based on the nearest neighbor join.

In [ ]:
buildings["street_index"] = momepy.get_nearest_street(
    buildings, streets, max_distance=100
)
buildings

Aattach the network index to the tessellation as well.

In [ ]:
tessellation["street_index"] = buildings["street_index"]

## Measure

Measure individual morphometric characters. For details see the User Guide and the API reference.

### Dimensions

In [ ]:
buildings["building_area"] = buildings.area
tessellation["tess_area"] = tessellation.area
streets["length"] = streets.length

### Shape

In [ ]:
buildings["eri"] = momepy.equivalent_rectangular_index(buildings)
buildings["elongation"] = momepy.elongation(buildings)
tessellation["convexity"] = momepy.convexity(tessellation)
streets["linearity"] = momepy.linearity(streets)

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(24, 12))

# buildings.plot("eri", ax=ax[0], scheme="natural_breaks", legend=True)
# buildings.plot("elongation", ax=ax[1], scheme="natural_breaks", legend=True)

# ax[0].set_axis_off()
# ax[1].set_axis_off()

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(24, 12))

# tessellation.plot("convexity", ax=ax[0], scheme="natural_breaks", legend=True)
# streets.plot("linearity", ax=ax[1], scheme="natural_breaks", legend=True)

# ax[0].set_axis_off()
# ax[1].set_axis_off()

### Spatial distribution

In [ ]:
buildings["shared_walls"] = momepy.shared_walls(buildings) / buildings.length
buildings.plot(
    "shared_walls", figsize=(12, 12), scheme="natural_breaks", legend=True
).set_axis_off()

Generate spatial graph using `libpysal`.

In [ ]:
queen_1 = libpysal.graph.Graph.build_contiguity(tessellation, rook=False)

In [ ]:
queen_1

In [ ]:
queen_1 = libpysal.graph.Graph.build_fuzzy_contiguity(tessellation, buffer=.5)
queen_1

In [ ]:
tessellation["neighbors"] = momepy.neighbors(
    tessellation, queen_1, weighted=True
)
tessellation["covered_area"] = queen_1.describe(tessellation.area)["sum"]
buildings["neighbor_distance"] = momepy.neighbor_distance(buildings, queen_1)

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(24, 12))

# buildings.plot(
#     "neighbor_distance", ax=ax[0], scheme="natural_breaks", legend=True
# )
# tessellation.plot(
#     "covered_area", ax=ax[1], scheme="natural_breaks", legend=True
# )

# ax[0].set_axis_off()
# ax[1].set_axis_off()

In [ ]:
queen_3 = queen_1.higher_order(3)
buildings_q1 = libpysal.graph.Graph.build_contiguity(buildings, rook=False)

buildings["interbuilding_distance"] = momepy.mean_interbuilding_distance(
    buildings, queen_1, queen_3
)
buildings["adjacency"] = momepy.building_adjacency(buildings_q1, queen_3)

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(24, 12))

# buildings.plot(
#     "interbuilding_distance", ax=ax[0], scheme="natural_breaks", legend=True
# )
# buildings.plot("adjacency", ax=ax[1], scheme="natural_breaks", legend=True)

# ax[0].set_axis_off()
# ax[1].set_axis_off()

In [ ]:
profile = momepy.street_profile(streets, buildings)
streets[profile.columns] = profile

In [ ]:
# fig, ax = plt.subplots(1, 3, figsize=(24, 12))

# streets.plot("width", ax=ax[0], scheme="natural_breaks", legend=True)
# streets.plot("width_deviation", ax=ax[1], scheme="natural_breaks", legend=True)
# streets.plot("openness", ax=ax[2], scheme="natural_breaks", legend=True)

# ax[0].set_axis_off()
# ax[1].set_axis_off()
# ax[2].set_axis_off()

### Intensity

In [ ]:
tessellation["car"] = buildings.area / tessellation.area
# tessellation.plot(
#     "car", figsize=(12, 12), vmin=0, vmax=1, legend=True
# ).set_axis_off()

### Connectivity

In [ ]:
graph = momepy.gdf_to_nx(streets)
graph = momepy.node_degree(graph)
graph = momepy.closeness_centrality(graph, radius=400, distance="mm_len")
graph = momepy.meshedness(graph, radius=400, distance="mm_len")
nodes, edges = momepy.nx_to_gdf(graph)

In [ ]:
# fig, ax = plt.subplots(1, 3, figsize=(24, 12))

# nodes.plot(
#     "degree", ax=ax[0], scheme="natural_breaks", legend=True, markersize=1
# )
# nodes.plot(
#     "closeness",
#     ax=ax[1],
#     scheme="natural_breaks",
#     legend=True,
#     markersize=1,
#     legend_kwds={"fmt": "{:.6f}"},
# )
# nodes.plot(
#     "meshedness", ax=ax[2], scheme="natural_breaks", legend=True, markersize=1
# )

# ax[0].set_axis_off()
# ax[1].set_axis_off()
# ax[2].set_axis_off()

In [ ]:
buildings["edge_index"] = momepy.get_nearest_street(buildings, edges)
buildings["node_index"] = momepy.get_nearest_node(
    buildings, nodes, edges, buildings["edge_index"]
)

Link all data together (to tessellation cells or buildings).

In [ ]:
tessellation.head()

In [ ]:
buildings.head()

In [ ]:
tessellation[buildings.columns.drop(["geometry", "street_index"])] = (
    buildings.drop(columns=["geometry", "street_index"])
)
merged = tessellation.merge(
    edges.drop(columns="geometry"),
    left_on="edge_index",
    right_index=True,
    how="left",
)
merged = merged.merge(
    nodes.drop(columns="geometry"),
    left_on="node_index",
    right_index=True,
    how="left",
)

In [ ]:
merged.columns

In [ ]:
attr_columns = merged.columns.drop(
    [
        "street_index",
        "node_index",
        "edge_index",
        "nodeID",
        "mm_len",
        "node_start",
        "node_end",
        "geometry",
        "x",
        "y",
        "id",
        "class",
        "_status",
        "landuse"
        # "shared_walls",
        # "adjacency"
    ]
)

In [ ]:
merged[attr_columns]

In [ ]:
attr_columns

In [ ]:
percentiles = []
for column in attr_columns:
    perc = momepy.percentile(merged[column], queen_3)
    perc.columns = [f"{column}_" + str(x) for x in perc.columns]
    percentiles.append(perc)

In [ ]:
percentiles_joined = pd.concat(percentiles, axis=1)
percentiles_joined.head()

In [ ]:
from spopt.region import SA3
import numpy
from sklearn.preprocessing import StandardScaler

In [ ]:
attr_columns

In [ ]:
# res = StandardScaler().fit_transform(merged[attr_columns])
# standardised_data = pd.DataFrame(res, columns=attr_columns).fillna(0)

In [ ]:
res = StandardScaler().fit_transform(percentiles_joined)
standardised_data = pd.DataFrame(res, columns=percentiles_joined.columns).fillna(0)

In [ ]:
clusterer = SA3(
    standardised_data,
    queen_1,
    standardised_data.columns,
    min_cluster_size=30,
    extraction="leaf",
)
clusterer.solve()
clusterer.labels_.value_counts()

In [ ]:
# queen_1.explore(gdf=tessellation)

In [ ]:
# buildings[["geometry"]].explore(
#     column=clusterer.labels_, categorical=True, legend=False
# )

## Clustering

Now we can use obtained values within a cluster analysis that should detect types of urban structure.

Standardize values before clustering.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

In [ ]:
grouped_data = standardised_data.groupby(clusterer.labels_).median()
if -1 in grouped_data.index:
    grouped_data = grouped_data.iloc[1:]

In [ ]:
linkage_matrix = linkage(grouped_data, method='complete')

In [ ]:
_ = dendrogram(linkage_matrix)

In [ ]:
morphotope_labels = fcluster(linkage_matrix, t=15, criterion='distance')
pd.Series(morphotope_labels).value_counts()

In [ ]:
final_labels = clusterer.labels_.replace(pd.Series(morphotope_labels, index=grouped_data.index).to_dict())
final_labels

In [ ]:
# baconsfield road is a problem

In [ ]:
buildings[["geometry"]].explore(
    column=final_labels, categorical=True, legend=True
)

In [ ]:
merged[attr_columns].groupby(final_labels).median().T.iloc[:, 1:].style.background_gradient(axis=1)